# Stack And Split Variables Across Files

Many solvers store a vector field as one file per component (`ux-0001.bin`, `uy-0001.bin`, `uz-0001.bin`) and a set of scalar fractions as numbered variables (`phi1-0001.bin`, `phi2-0001.bin`). In memory, you would rather work with one array `u` carrying a component dimension `i`, and one array `phi` carrying a fraction dimension `n`.

xarray-binfile describes that encoding once, with a `VariableStack`, and uses it in both directions:

- **split** on write: `u` with `i = ["x", "y", "z"]` becomes the variables `ux`, `uy` and `uz`, each written to its own files
- **stack** on read: `ux`, `uy` and `uz` found on disk are concatenated back into `u`

Nothing about `u`, `x`, `y`, `z` or `phi` is hard-coded in the library: the dimension name, the name template and the allowed values all come from the declaration.

In [ ]:
import pathlib
import tempfile

import numpy as np
import xarray as xr

import xarray_binfile  # noqa: F401  (registers the .binary_engine accessors)
from xarray_binfile.conventions import Layout, StepIndexedFiles, VariableStack

directory_holder = tempfile.TemporaryDirectory()
directory = pathlib.Path(directory_holder.name)

## Declare the stacks

A `VariableStack` has three parts:

- `dim`: the dimension that lives in the variable names rather than in the files
- `template`: how a variable name is built from the base `{name}` and the value of `dim`
- `values`: the values the dimension may take, in the order the stacked coordinate should follow

The template is parsed with `FilenamePattern`, so integer values can carry zero-fill (`"{name}{n:02d}"`) and are converted back to `int` on read. `values` is required because templates without a separator would otherwise match almost any name: `pp` fits `"{name}{i}"` as `p` + `p`.

In [ ]:
velocity = VariableStack("i", "{name}{i}", values=("x", "y", "z"))
scalars = VariableStack("n", "{name}{n:d}", values=range(1, 10), names=("phi",))
velocity, scalars

The optional `names` restricts stacking to given base names. Without it, a vorticity component written as `w3` would be picked up by `scalars` as `w` with `n = [3]`, which is harmless but probably not what you want.

## Attach them to a convention

Every shipped convention accepts `stacks`. On write, arrays are split in the order the stacks are listed, before the per-file split along `time`; on read, `open` rebuilds the stacked arrays by default.

In [ ]:
layout = Layout(
    {
        "x": np.linspace(0.0, 1.0, 8, dtype=np.float32),
        "y": np.linspace(0.0, 2.0, 6, dtype=np.float32),
        "z": np.linspace(0.0, 0.5, 4, dtype=np.float32),
    },
    dtype=np.float32,
)
convention = StepIndexedFiles(
    layout,
    pattern="{name}-{step:03d}.bin",
    time_step=0.25,
    stacks=[velocity, scalars],
)
convention

## Build a dataset with stacked arrays

`u` carries `i`, `phi` carries `n`, and `pp` carries neither. All of them share the mesh and the `time` dimension.

In [ ]:
rng = np.random.default_rng(0)
time = np.arange(3) * 0.25


def field(**extra):
    coords = {**extra, **layout.coords, "time": time}
    shape = tuple(len(values) for values in coords.values())
    return xr.DataArray(rng.random(shape, dtype=np.float32), coords=coords)


dataset = xr.Dataset(
    {
        "u": field(i=["x", "y", "z"]),
        "phi": field(n=[1, 2]),
        "pp": field(),
    }
)
dataset

## Split on write

Writing the dataset through the convention produces one file per component, per fraction and per step. `u` never appears on disk; `pp` is written as is.

In [ ]:
dataset.binary_engine.to_file(convention.writer, directory)
sorted(path.name for path in directory.glob("*.bin"))

The split happens before the file split, so the per-file slices are plain `(x, y, z)` arrays. You can inspect the plan without writing anything by iterating the writer:

In [ ]:
[(spec.filename, spec.sub_array.dims) for spec in convention.writer(dataset["u"])][:4]

## Stack on read

`open` lists the files that follow the pattern, opens them lazily with `xarray.open_mfdataset`, and applies the stacks. The result has the same variables as the original dataset, with `i` and `n` as proper coordinates.

In [ ]:
opened = convention.open(directory, chunks={"time": 1})
opened

In [ ]:
for name in dataset.data_vars:
    xr.testing.assert_allclose(
        opened[name].transpose(*dataset[name].dims).load(), dataset[name]
    )
print("round trip matches")

Stacking is lazy, so a Dask-backed dataset stays Dask-backed, and `i` and `n` can be used like any other coordinate:

In [ ]:
speed = np.sqrt((opened["u"] ** 2).sum("i")).rename("speed")
speed.isel(time=0, z=0).compute()

## Opt out, or stack by hand

Pass `stack=False` to see the variables exactly as they are on disk. The same declarations are available as functions for datasets opened another way: `VariableStack.stack` for one dimension, `stack_variables` for a sequence, and `split_variables` for the inverse.

In [ ]:
raw = convention.open(directory, stack=False)
sorted(raw.data_vars)

In [ ]:
from xarray_binfile.conventions import split_variables, stack_variables

restacked = stack_variables(raw, [velocity, scalars])
pieces = [piece.name for piece in split_variables(dataset["u"], [velocity])]
sorted(restacked.data_vars), pieces

## Guard rails

A value that is not declared cannot be written, because the reader would have no way to stack it back:

In [ ]:
bad = dataset["u"].assign_coords(i=["x", "y", "w"])
try:
    list(convention.writer(bad))
except ValueError as error:
    print(error)

In [ ]:
directory_holder.cleanup()